# Baseline matrix label analysis

Viewer for `scripts/baseline_matrix_label_analysis.py`. Logic lives in script; notebook only launches/reads cache and plots.

In [ ]:
from pathlib import Path
import os


def project_root(start=Path.cwd()):
    start = Path(start).resolve()
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists():
            return path
    raise FileNotFoundError("pyproject.toml not found")

os.chdir(project_root())
Path.cwd()

In [ ]:
PROBE_ROOT = Path("runs/baseline_matrix_probe_mean_100ep+100ep_medium")
OUTPUT = PROBE_ROOT / "label_analysis"
CACHE = OUTPUT / "cache.npz"
OUTPUT.mkdir(parents=True, exist_ok=True)

PROBE_ROOT, OUTPUT, CACHE

## Build cache

Run once. Uses CUDA by default. Re-run with `--force` if probe checkpoints or moments settings changed.

In [ ]:
import subprocess

if not CACHE.exists():
    subprocess.run([
        "uv", "run", "python", "scripts/baseline_matrix_label_analysis.py",
        "--probe-root", str(PROBE_ROOT),
        "--output", str(OUTPUT),
        "--cache", str(CACHE),
        "--passes", "4",
        "--device", "cuda",
    ], check=True)
else:
    print(f"using cached {CACHE}")

In [ ]:
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

data = dict(np.load(CACHE, allow_pickle=True))
inputs = [str(x) for x in data["inputs"]]
losses = [str(x) for x in data["losses"]]
metrics = [str(x) for x in data["metrics"]]

probe_ap = data["probe_ap"].astype(float)
moments_ap = data["moments_ap"].astype(float)
probe_scores = data["probe_scores"].astype(float)
moments_scores = data["moments_scores"].astype(float)
lift = probe_ap - moments_ap[:, None, :]
macro_lift = np.nanmean(lift, axis=2)

def metric(name):
    return probe_scores[:, :, metrics.index(name)]

def show_matrix(values, title, fmt=".4f"):
    frame = pl.DataFrame({"input": inputs, **{loss: values[:, j] for j, loss in enumerate(losses)}})
    print(title)
    display(frame)

def heatmap(values, title, cmap="viridis", center_zero=False):
    fig, ax = plt.subplots(figsize=(4.8, 3.6))
    kwargs = {}
    if center_zero:
        vmax = np.nanmax(np.abs(values))
        kwargs.update(vmin=-vmax, vmax=vmax)
    im = ax.imshow(values, cmap=cmap, **kwargs)
    ax.set_xticks(range(len(losses)), losses)
    ax.set_yticks(range(len(inputs)), inputs)
    ax.set_xlabel("loss channels")
    ax.set_ylabel("input channels")
    ax.set_title(title)
    for i in range(values.shape[0]):
        for j in range(values.shape[1]):
            ax.text(j, i, f"{values[i, j]:.3f}", ha="center", va="center", color="white" if values[i,j] < np.nanmean(values) else "black")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    return fig

## Overview report

In [ ]:
report = OUTPUT / "overview.md"
if report.exists():
    display(Markdown(report.read_text()))
else:
    print("overview.md missing; run script cell above")

## Matrix heatmaps

In [ ]:
show_matrix(metric("val/macro_map"), "macro mAP")
show_matrix(metric("val/micro_map"), "micro mAP")
show_matrix(macro_lift, "macro lift over matching moments")

heatmap(metric("val/macro_map"), "macro mAP")
heatmap(macro_lift, "macro AP lift over matching moments", cmap="coolwarm", center_zero=True)

## Saved figures

Generated by script in `OUTPUT/images`.


In [ ]:
from IPython.display import Image, display

for name in [
    "group_summary",
    "gap_dumbbell",
    "scatter",
    "coverage_null",
    "macro_map_heatmap",
    "macro_lift_heatmap",
    "all_cells_lift_heatmap",
    "winner_counts",
    "all_cells_dumbbell",
]:
    path = OUTPUT / "images" / f"{name}.png"
    if path.exists():
        print(name)
        display(Image(filename=str(path)))
    else:
        print("missing", path)


## Per-label table

In [ ]:
labels = pl.read_csv(OUTPUT / "per_label.csv")
summary = pl.read_csv(OUTPUT / "summary_metrics.csv")
moments = pl.read_csv(OUTPUT / "moments_metrics.csv")

display(summary.sort("val/macro_map", descending=True))
display(moments.sort("macro_map_from_labels", descending=True))
labels.head()

## Best cell vs matching moments

In [ ]:
best = summary.sort("val/macro_map", descending=True).row(0, named=True)
best_cell = best["cell"]
best_input = best["input"]
probe_col = f"probe_{best_cell}"
lift_col = f"lift_{best_cell}"
mom_col = f"moments_{best_input}"

print(best_cell, "vs", mom_col)

fig, ax = plt.subplots(figsize=(6, 6))
x = labels[mom_col].to_numpy()
y = labels[probe_col].to_numpy()
valid = np.isfinite(x) & np.isfinite(y)
ax.scatter(x[valid], y[valid], s=22, alpha=0.75)
limit = max(float(np.nanmax(x)), float(np.nanmax(y))) * 1.05
ax.plot([0, limit], [0, limit], "--", color="0.4", lw=1)
ax.set_xlabel(mom_col)
ax.set_ylabel(probe_col)
ax.set_title(f"{best_cell} per-label AP vs matching moments")
ax.grid(alpha=0.25)
fig.tight_layout()

## Top gained / lost labels

In [ ]:
display(
    labels.select("label", "positives", "coverage", "sequence_share", mom_col, probe_col, lift_col)
    .sort(lift_col, descending=True)
    .head(15)
)

display(
    labels.select("label", "positives", "coverage", "sequence_share", mom_col, probe_col, lift_col)
    .sort(lift_col)
    .head(15)
)

## Label search

Set `QUERY` to inspect one label or substring.

In [ ]:
QUERY = "jump"
labels.filter(pl.col("label").str.contains(QUERY, literal=False))